In [ ]:
# Training Loop with Progress Tracking
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

def train_model(model, dataset, training_args):
    # Create data loader
    train_dataloader = DataLoader(
        dataset, 
        batch_size=training_args.per_device_train_batch_size,
        shuffle=True
    )
    
    # Initialize optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=training_args.learning_rate)
    
    # Training loop
    model.train()
    for epoch in range(training_args.num_train_epochs):
        print(f"Epoch {epoch + 1}/{training_args.num_train_epochs}")
        progress_bar = tqdm(train_dataloader)
        
        for batch in progress_bar:
            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss
            
            # Backward pass
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            # Update progress bar
            progress_bar.set_description(f"Loss: {loss.item():.4f}")

In [ ]:
# Save Quantized Model
def save_quantized_model(model, tokenizer, output_dir):
    # Save model in 4-bit quantization
    model.save_pretrained(
        output_dir,
        save_4bit=True,
        use_safetensors=True
    )
    
    # Save tokenizer
    tokenizer.save_pretrained(output_dir)
    
    # Save quantization configuration
    quantization_config = {
        "load_in_4bit": True,
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True
    }
    
    with open(f"{output_dir}/quantization_config.json", "w") as f:
        json.dump(quantization_config, f)

In [ ]:
# Evaluation Functions
from sklearn.metrics import accuracy_score
import numpy as np

def evaluate_model(model, eval_dataset):
    model.eval()
    predictions = []
    references = []
    
    eval_dataloader = DataLoader(eval_dataset, batch_size=8)
    
    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc="Evaluating"):
            outputs = model(**batch)
            logits = outputs.logits
            
            # Get predictions
            pred = torch.argmax(logits, dim=-1)
            predictions.extend(pred.cpu().numpy())
            references.extend(batch["labels"].cpu().numpy())
    
    # Calculate metrics
    metrics = {
        "accuracy": accuracy_score(references, predictions),
        "perplexity": torch.exp(outputs.loss).item()
    }
    
    return metrics

In [ ]:
# Main training and evaluation
if __name__ == "__main__":
    # Train model
    train_metrics = train_model(model, dataset, training_args)
    
    # Evaluate
    eval_metrics = evaluate_model(model, dataset)
    print("Evaluation metrics:", eval_metrics)
    
    # Save model
    save_quantized_model(
        model, 
        tokenizer, 
        "./process_mining_model_quantized"
    )